In [ ]:
import numpy as np
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import torch.optim as optim
import cv2
import pickle
from matplotlib.gridspec import GridSpec
from tqdm import tqdm
import os
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def denorm(p, size=960):
    result = np.zeros_like(p)
    result[..., 0] = (p[..., 0] + 1) * (size / 2)
    result[..., 1] = size - ((p[..., 1] + 1) * (size / 2))
    return result

In [ ]:
map=cv2.imread('Baarle-Nassau_-_Baarle-Hertog-en no legend.png')[:,:,(2,1,0)]

belgium_color=np.array([251,234,81])
netherlands_color=np.array([255, 255, 228])

netherlands_region=((map-netherlands_color)**2).sum(-1)<50
belgium_region=((map-belgium_color)**2).sum(-1)<10000


b_coords=np.array(np.where(belgium_region)).T.astype('float')
n_coords=np.array(np.where(netherlands_region)).T.astype('float')

#flip and normalize
belgium_coords_all=np.zeros_like(b_coords)
netherlands_coords_all=np.zeros_like(n_coords)

belgium_coords_all[:,0]=b_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
belgium_coords_all[:,1]=(960-b_coords[:,0])/(960/2)-1
netherlands_coords_all[:,0]=n_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
netherlands_coords_all[:,1]=(960-n_coords[:,0])/(960/2)-1

## Hmm to balance to classes or do more area proportional? Not sure yet. 
np.random.seed(55)
num_points_to_sample=10000 #10000
belgium_coords_sample=belgium_coords_all[np.random.choice(len(belgium_coords_all), num_points_to_sample), :]
netherlands_coords_sample=netherlands_coords_all[np.random.choice(len(netherlands_coords_all), num_points_to_sample), :]
X_sample=np.vstack((netherlands_coords_sample, belgium_coords_sample))
y_sample=np.concatenate((np.zeros(len(belgium_coords_sample)), np.ones(len(netherlands_coords_sample)))).astype('int')
rI=np.arange(len(y_sample))
np.random.shuffle(rI)
X_sample=X_sample[rI,:]
y_sample=y_sample[rI]
X_sample = torch.FloatTensor(X_sample)
y_sample = torch.tensor(y_sample)

#Skip sampling train on all. 
belgium_coords=belgium_coords_all
netherlands_coords=netherlands_coords_all

X=np.vstack((netherlands_coords, belgium_coords))
y=np.concatenate((np.zeros(len(netherlands_coords)), np.ones(len(belgium_coords)))).astype('int')

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

In [ ]:
class BaarleNet(nn.Module):
    def __init__(self, hidden_layers=[64]):
        super(BaarleNet, self).__init__()
        layers = [nn.Linear(2, hidden_layers[0]), nn.ReLU()]
        for i in range(len(hidden_layers)-1):
            layers.append(nn.Linear(hidden_layers[i], hidden_layers[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_layers[-1], 2))
        self.layers=layers
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

In [ ]:
def viz_descision_boundary(model, res=256):
    plt.clf()
    fig=plt.figure(0,(6,6))
    ax=fig.add_subplot(111)
    
    probe=np.zeros((res,res,2))
    for j, xx in enumerate(np.linspace(-1, 1, res)):
        for k, yy in enumerate(np.linspace(-1, 1, res)):
            probe[j, k]=[yy,xx]
    probe=probe.reshape(res**2, -1)
    probe_logits=model(torch.tensor(probe).float())
    probe_logits=probe_logits.detach().numpy().reshape(res,res,2)
    probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))
    
    ax.imshow(map.mean(2), cmap='gray')
    ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
               extent=[0, 960, 960, 0],  # This maps to image coordinates
               alpha=0.7,
               cmap='viridis')


## [2]

In [ ]:
plt.scatter(X_sample[:,0], X_sample[:,1], c=y_sample, s=1, alpha=0.3); plt.axis('square');

In [ ]:
X_sample.shape

In [ ]:
random_seed=1
num_epochs = 1
batch_size = 1

device='cpu'

torch.manual_seed(random_seed)  

# Initialize model and optimizer
model = BaarleNet([2]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    #'gradients_history': [],
    #'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

num_epochs = 2500
for epoch in range(num_epochs):

    #Stochastic
    outputs = model(X_sample[epoch%len(y_sample)])
    loss = criterion(outputs, y_sample[epoch%len(y_sample)])
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    
    # Store model weights (deep copy to avoid reference issues)
    current_weights = {}
    for name, param in model.named_parameters():
        current_weights[name] = param.data.clone()
    training_data['weights_history'].append(current_weights)
    
    # Store loss
    training_data['loss_history'].append(loss.item())
    
    optimizer.step()
    if (epoch+1) % (num_epochs//25) == 0:
        outputs_batch = model(X_sample)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_sample).sum().item()/len(y_sample)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
viz_descision_boundary(model)

In [ ]:
output_dir='data/backprop3/training_caches'
os.makedirs(output_dir, exist_ok=True)
filename = os.path.join(output_dir, '2_1.pkl')
with open(filename, 'wb') as f:
    pickle.dump(training_data, f)

## [3]

In [ ]:
y_sample[0]

In [ ]:
X_sample[0].numpy().tolist()

In [ ]:
# X_sample[epoch%len(y_sample)]).numpy().list()

In [ ]:
random_seed=15
num_epochs = 1
batch_size = 1

device='cpu'

torch.manual_seed(random_seed)  

# Initialize model and optimizer
model = BaarleNet([3]).to(device)

bound=2.5
with torch.no_grad():
    model.model[0].weight.uniform_(-bound, bound)
    model.model[0].bias.uniform_(-bound, bound)  
    model.model[2].weight.uniform_(-bound, bound)
    model.model[2].bias.uniform_(-bound, bound)  


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    'gradients_history': [],
    'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

num_epochs = 2500
for epoch in range(num_epochs):

    #Stochastic
    outputs = model(X_sample[epoch%len(y_sample)])
    loss = criterion(outputs, y_sample[epoch%len(y_sample)])
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # Store current X,y values
    training_data['xy_history'].append({
        'epoch': epoch,
        'x': X_sample[epoch%len(y_sample)].numpy().tolist(),
        'y':  y_sample[epoch%len(y_sample)].item()
    })

    current_gradients = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            current_gradients[name] = param.grad.clone()
        else:
            current_gradients[name] = None
    training_data['gradients_history'].append(current_gradients)

    # Store model weights (deep copy to avoid reference issues)
    current_weights = {}
    for name, param in model.named_parameters():
        current_weights[name] = param.data.clone()
    training_data['weights_history'].append(current_weights)
    
    # Store loss
    training_data['loss_history'].append(loss.item())
    
    optimizer.step()
    if (epoch+1) % (num_epochs//25) == 0:
        outputs_batch = model(X_sample)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_sample).sum().item()/len(y_sample)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
viz_descision_boundary(model)

In [ ]:
# output_dir='data/backprop3/training_caches'
# os.makedirs(output_dir, exist_ok=True)
# filename = os.path.join(output_dir, '3_2.pkl')
# with open(filename, 'wb') as f:
#     pickle.dump(training_data, f)

## [4]

## [2 2]

## [3 3]

## [8 8]

In [ ]:
random_seed=25
# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    #'gradients_history': [],
    #'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

In [ ]:
device='cpu'
batch_size = 10000
torch.manual_seed(random_seed)  
X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Create dataset and dataloader for minibatching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model and optimizer
model = BaarleNet([8, 8]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) #Try a little higher lr, see if we can sped things up. 0.005 worked really well
num_epochs = 1000

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_X.to(device))
        loss = criterion(outputs, batch_y.to(device))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
    
        # Store model weights (deep copy to avoid reference issues)
        if epoch<20:
            current_weights = {}
            for name, param in model.named_parameters():
                current_weights[name] = param.data.clone()
            training_data['weights_history'].append(current_weights)
            
            # Store loss
            training_data['loss_history'].append(loss.item())
    
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1

    if epoch>20:
        current_weights = {}
        for name, param in model.named_parameters():
            current_weights[name] = param.data.clone()
        training_data['weights_history'].append(current_weights)
        
        # Store loss
        training_data['loss_history'].append(loss.item())

    # Calculate average loss for the epoch
    avg_loss = epoch_loss / num_batches
    
    if (epoch + 1) % (num_epochs // 100) == 0:
        # Calculate accuracy on full dataset
        with torch.no_grad():
            outputs_batch = model(X_sample.to(device))
            accuracy = (torch.argmax(outputs_batch, dim=1) == y_sample.to(device)).sum().item() / len(y_sample)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
viz_descision_boundary(model)

In [ ]:
output_dir='data/backprop3/training_caches'
os.makedirs(output_dir, exist_ok=True)
filename = os.path.join(output_dir, '8_8_1.pkl')
with open(filename, 'wb') as f:
    pickle.dump(training_data, f)

## [8 8 8]

In [ ]:
random_seed = 4
# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    #'gradients_history': [],
    #'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

device='cpu'
batch_size = 10000
torch.manual_seed(random_seed)  
X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Create dataset and dataloader for minibatching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model and optimizer
model = BaarleNet([8, 8, 8]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.02) #Try a little higher lr, see if we can sped things up. 0.005 worked really well
num_epochs = 30

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_X.to(device))
        loss = criterion(outputs, batch_y.to(device))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
    
        # Store model weights (deep copy to avoid reference issues)
        # if epoch<20:
        current_weights = {}
        for name, param in model.named_parameters():
            current_weights[name] = param.data.clone()
        training_data['weights_history'].append(current_weights)
        
        # Store loss
        training_data['loss_history'].append(loss.item())
    
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1

    # if epoch>20:
    #     current_weights = {}
    #     for name, param in model.named_parameters():
    #         current_weights[name] = param.data.clone()
    #     training_data['weights_history'].append(current_weights)
        
    #     # Store loss
    #     training_data['loss_history'].append(loss.item())

    # Calculate average loss for the epoch
    avg_loss = epoch_loss / num_batches
    
    # Calculate accuracy on full dataset
    with torch.no_grad():
        outputs_batch = model(X_sample.to(device))
        accuracy = (torch.argmax(outputs_batch, dim=1) == y_sample.to(device)).sum().item() / len(y_sample)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
viz_descision_boundary(model)

In [ ]:
output_dir='data/backprop3/training_caches'
os.makedirs(output_dir, exist_ok=True)
filename = os.path.join(output_dir, '8_8_8_2.pkl')
with open(filename, 'wb') as f:
    pickle.dump(training_data, f)

## [16 16 ]

## [16 16 16]

In [ ]:
random_seed = 25
# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    #'gradients_history': [],
    #'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

device='cpu'
batch_size = 10000
torch.manual_seed(random_seed)  
X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Create dataset and dataloader for minibatching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model and optimizer
model = BaarleNet([16, 16, 16]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) #Try a little higher lr, see if we can sped things up. 0.005 worked really well
num_epochs = 60

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_X.to(device))
        loss = criterion(outputs, batch_y.to(device))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
    
        # Store model weights (deep copy to avoid reference issues)
        # if epoch<20:
        current_weights = {}
        for name, param in model.named_parameters():
            current_weights[name] = param.data.clone()
        training_data['weights_history'].append(current_weights)
        
        # Store loss
        training_data['loss_history'].append(loss.item())
    
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1

    # if epoch>20:
    #     current_weights = {}
    #     for name, param in model.named_parameters():
    #         current_weights[name] = param.data.clone()
    #     training_data['weights_history'].append(current_weights)
        
    #     # Store loss
    #     training_data['loss_history'].append(loss.item())

    # Calculate average loss for the epoch
    avg_loss = epoch_loss / num_batches
    
    # Calculate accuracy on full dataset
    with torch.no_grad():
        outputs_batch = model(X_sample.to(device))
        accuracy = (torch.argmax(outputs_batch, dim=1) == y_sample.to(device)).sum().item() / len(y_sample)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
viz_descision_boundary(model)

In [ ]:
output_dir='data/backprop3/training_caches'
os.makedirs(output_dir, exist_ok=True)
filename = os.path.join(output_dir, '16_16_16_1.pkl')
with open(filename, 'wb') as f:
    pickle.dump(training_data, f)

## [32 32 32 32]

In [ ]:
# Data storage for this run
training_data = {
    'seed': random_seed,
    'weights_history': [],
    #'gradients_history': [],
    #'xy_history': [],
    'loss_history': [],
    #'accuracy_history': []
}

In [ ]:
device='cpu'
batch_size = 10000
random_seed=25

torch.manual_seed(random_seed)  
X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Create dataset and dataloader for minibatching
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model and optimizer
model = BaarleNet([32, 32, 32, 32]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) #Try a little higher lr, see if we can sped things up. 0.005 worked really well
num_epochs = 1000

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_X.to(device))
        loss = criterion(outputs, batch_y.to(device))
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
    
        # Store model weights (deep copy to avoid reference issues)
        if epoch<20:
            current_weights = {}
            for name, param in model.named_parameters():
                current_weights[name] = param.data.clone()
            training_data['weights_history'].append(current_weights)
            
            # Store loss
            training_data['loss_history'].append(loss.item())
    
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1

    if epoch>20:
        current_weights = {}
        for name, param in model.named_parameters():
            current_weights[name] = param.data.clone()
        training_data['weights_history'].append(current_weights)
        
        # Store loss
        training_data['loss_history'].append(loss.item())

    # Calculate average loss for the epoch
    avg_loss = epoch_loss / num_batches
    
    if (epoch + 1) % (num_epochs // 100) == 0:
        # Calculate accuracy on full dataset
        with torch.no_grad():
            outputs_batch = model(X_sample.to(device))
            accuracy = (torch.argmax(outputs_batch, dim=1) == y_sample.to(device)).sum().item() / len(y_sample)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

In [ ]:
len(dataloader)

In [ ]:
viz_descision_boundary(model)

In [ ]:
output_dir='data/backprop3/training_caches'
os.makedirs(output_dir, exist_ok=True)
filename = os.path.join(output_dir, '32_32_32_32_1.pkl')
with open(filename, 'wb') as f:
    pickle.dump(training_data, f)